In [1]:
import pandas as pd
import matplotlib.pyplot as plt
# Use only Domestic/Men's/IPL/2022-2025 data

In [2]:
datas = [pd.read_csv(f'../data/Domestic/Men\'s/IPL/{y}/ipl_{y}_deliveries.csv') for y in range(2022, 2026)]
data = pd.concat(datas, ignore_index=True)
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data.info()
data.sample(frac=1).reset_index(drop=True).head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69597 entries, 0 to 69596
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   match_id          69597 non-null  int64         
 1   season            69597 non-null  int64         
 2   match_no          69597 non-null  int64         
 3   date              69597 non-null  datetime64[ns]
 4   venue             69597 non-null  object        
 5   batting_team      69597 non-null  object        
 6   bowling_team      69597 non-null  object        
 7   innings           69597 non-null  int64         
 8   over              69597 non-null  float64       
 9   striker           69597 non-null  object        
 10  bowler            69597 non-null  object        
 11  runs_of_bat       69597 non-null  int64         
 12  extras            69597 non-null  int64         
 13  wide              69597 non-null  int64         
 14  legbyes           6959

,match_id,season,match_no,date,venue,batting_team,bowling_team,innings,over,striker,...,runs_of_bat,extras,wide,legbyes,byes,noballs,wicket_type,player_dismissed,fielder,phase
0,202411,2024,11,2024-03-30,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,LSG,PBKS,1,9.5,Pooran,...,1,0,0,0,0,0,NaN,NaN,NaN,NaN
1,202316,2023,16,2023-04-11,"Arun Jaitley Stadium, Delhi",DC,MI,1,12.5,Axar,...,1,0,0,0,0,0,NaN,NaN,NaN,NaN
2,202444,2024,44,2024-04-27,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,RR,LSG,2,2.6,Buttler,...,1,0,0,0,0,0,NaN,NaN,NaN,NaN
3,202373,2023,73,2023-05-26,"Narendra Modi Stadium, Ahmedabad",MI,GT,2,9.1,Suryakumar Yadav,...,2,0,0,0,0,0,NaN,NaN,NaN,NaN
4,202462,2024,62,2024-05-12,"M.Chinnaswamy Stadium, Bengaluru",DC,RCB,2,7.1,Shai Hope,...,2,0,0,0,0,0,NaN,NaN,NaN,NaN


In [3]:
# Further exploration of data
# Print unique values for each column
for column in data.columns:
    unique_values = data[column].unique()
    print(f"Column: {column}, Unique Values: {len(unique_values)}")
    if len(unique_values) <= 20:
        print(f"Values: {unique_values}")
    print("-" * 50)

unique_matches = data['match_id'].unique()
unique_teams = pd.concat([data['batting_team'], data['bowling_team']]).unique()
unique_players = pd.concat([data['striker'], data['bowler']]).unique()

Column: match_id, Unique Values: 291
--------------------------------------------------
Column: season, Unique Values: 4
Values: [2022 2023 2024 2025]
--------------------------------------------------
Column: match_no, Unique Values: 74
--------------------------------------------------
Column: date, Unique Values: 239
--------------------------------------------------
Column: venue, Unique Values: 18
Values: ['Wankhede Stadium, Mumbai' 'Brabourne Stadium, Mumbai'
 'Dr DY Patil Sports Academy, Mumbai'
 'Maharashtra Cricket Association Stadium, Pune' 'Eden Gardens, Kolkata'
 'Narendra Modi Stadium, Ahmedabad'
 'Punjab Cricket Association IS Bindra Stadium, Mohali'
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow'
 'Rajiv Gandhi International Stadium, Hyderabad'
 'M.Chinnaswamy Stadium, Bengaluru' 'MA Chidambaram Stadium, Chennai'
 'Arun Jaitley Stadium, Delhi' 'Barsapara Cricket Stadium, Guwahati'
 'Sawai Mansingh Stadium, Jaipur'
 'Himachal Pradesh Cricket Assoc

Combine everything into a per-match per-player aggregate dataset to use for XGBoost model training. The final dataset will include the following:
## Player and match columns
- **player**: Name of the player
- **match_id**: Unique identifier for the match
- **match_date**: Date of the match
- **player_roles**: List of roles the player can fill (e.g., `['striker', 'bowler', 'all-rounder', 'wicket-keeper']`) (note: all-rounder is a player who can both bat and bowl)
- **fantasy_points**: Running average of total fantasy points scored by the player in the match (calculated based on the scoring system)
- **team_played**: Name of the team the player played for in the match
- **team_faced**: Name of the team the player faced in the match
- **venue**: Name of the venue where the match was played
- **pitch**: Type of pitch (e.g., `green`, `flat`, `dust`, `slow`)
#### Weather stats
- **temperature**: Temperature in Celsius during the match
- **humidity**: Humidity percentage during the match
- **wind_speed**: Average wind speed in km/h during the match
<!--
>    ### Batting Stats
>    - **score**: Total runs scored by the player in the match
>    - **balls_faced**: Total balls faced by the player in the match
>    - **fours**: Total number of fours hit by the player in the match
>    - **sixes**: Total number of sixes hit by the player in the match
>    - **out**: Whether the player was out in the match (True/False)
>    ### Bowling stats
>    - **balls_bowled**: Total balls bowled by the player in the match
>    - **runs_conceded**: Total runs conceded by the player in the match
>    - **wickets**: Total wickets taken by the player in the match
>    ### Fielding stats
>    - **catches**: Total number of catches taken by the player in the match
>    - **run_outs**: Total number of run outs effected by the player in the match
-->
### Running average Stats
- **matches_played**: Total number of matches played
#### Batting stats
- **average_score**: Average score (per ball)
- **total_balls_faced**: Total number of balls faced
- **total_outs**: Total number of times player was out in matches
- **total_fours**: Total number of fours hit
- **total_sixes**: Total number of sixes hit
- **matches_played_batting**: Total number of matches played where the player batted
#### Bowling stats
- **average_runs_conceded**: Average runs conceded by the player per match
- **total_balls_bowled**: Total balls bowled by the player across all matches
- **total_wickets_taken**: Total wickets taken by the player across all matches
- **matches_played_bowling**: Total number of matches played by the player (where the player bowled at least one ball)
#### Fielding stats
- **average_catches**: Average number of catches taken by the player per match
- **average_run_outs**: Average number of run outs effected by the player per match
- **matches_played_fielding**: Total number of matches played by the player (where the player fielded at least once)


## Our model will be predicting per player per match fantasy scores

In [4]:
def fantasy_points(deliveries, player):
    if 'match_id' in deliveries.columns:
        # We possible have multiple match data, return fantasy points for that match
        unique_matches = deliveries['match_id'].unique()
        if len(unique_matches) == 0:
            return 0
        if len(unique_matches) > 1:  # More than one match data, return average of each match's fantasy points
            return deliveries.groupby('match_id').apply(lambda x: fantasy_points(x, player), include_groups=False).mean()

    # Ensure deliveries are in chronological order
    deliveries = deliveries.sort_values(by=['innings', 'over'])
    f = 4  # Lineup points
    score = 0
    wickets = 0
    catches = 0
    over_score = 0
    for d in deliveries.itertuples():
        over_score += d.runs_of_bat
        if d.striker == player:
            f += d.runs_of_bat + (1 if d.runs_of_bat == 4 else 0) + (2 if d.runs_of_bat == 6 else 0)
            score += d.runs_of_bat
            if d.player_dismissed == player and score == 0:
                # TODO: no points are deducted if player is a bowler!!!
                f -= 2  # Deduct points for getting out for a duck
        if d.bowler == player:
            if not pd.isna(d.player_dismissed) and d.wicket_type not in ['runout', 'obstructing the field', 'retired out']:
                f += 25
                wickets += 1
        if d.fielder == player:
            f += (8 if d.wicket_type == 'caught' else 0) + (12 if d.wicket_type in ['runout', 'stumped'] else 0)
            catches += 1 if d.wicket_type == 'caught' else 0
        # Reset over
        if str(d.over).split('.')[1] == '6':
            if over_score == 0 and d.bowler == player:
                f += 12  # Add points for maiden over
            over_score = 0  # Reset score for the new over
    if wickets >= 3:
        f += 4 * (min(wickets, 5) - 2)  # Bonus points for taking 3-5 wickets: 4, 8, 12 points for 3, 4, 5 wickets respectively
    if score >= 50:
        f += 8
    if score >= 100:
        f += 16
    if catches >= 3:
        f += 4
    return f

In [ ]:
# Temporary df handling running averages
temp_df = []
# Get match dates in ascending order
for m in unique_matches:
    # get players that have played in this match
    date = data[data['match_id'] == m]['date'].iloc[0]
    players = data[data['match_id'] == m][['striker', 'bowler']].melt(value_name='player')['player'].unique()
    for player in players:
        fp = fantasy_points(data[data['match_id'] == m], player)

        # Calculate total stats across all matches played by the player up to this match
        num_matches = data[(data['date'] < date) & ((data['striker'] == player) | (data['bowler'] == player) | (data['fielder'] == player))]['match_id'].nunique()
        total_score = data[(data['date'] < date) & (data['striker'] == player)]['runs_of_bat'].sum()
        total_balls_faced = data[(data['date'] < date) & (data['striker'] == player)].shape[0]
        total_outs = data[(data['date'] < date) & (data['striker'] == player)]['player_dismissed'].notna().sum()
        total_fours = data[(data['date'] < date) & (data['striker'] == player) & (data['runs_of_bat'] == 4)].shape[0]
        total_sixes = data[(data['date'] < date) & (data['striker'] == player) & (data['runs_of_bat'] == 6)].shape[0]
        total_balls_bowled = data[(data['date'] < date) & (data['bowler'] == player)].shape[0]
        total_runs_conceded = data[(data['date'] < date) & (data['bowler'] == player)]['runs_of_bat'].sum() + data[(data['date'] < date) & (data['bowler'] == player)]['extras'].sum()
        total_wickets_taken = data[(data['date'] < date) & (data['bowler'] == player)]['player_dismissed'].notna().sum()
        total_catches = data[(data['date'] < date) & (data['fielder'] == player) & (data['wicket_type'] == 'caught')].shape[0]
        total_run_outs = data[(data['date'] < date) & (data['fielder'] == player) & (data['wicket_type'] == 'runout')].shape[0]
        total_fantasy_points = fantasy_points(data[data['date'] < date], player)
        total_venue_fantasy_points = fantasy_points(data[(data['date'] < date) & (data['venue'] == data[data['match_id'] == m]['venue'].iloc[0])], player)
        total_against_opposition_fantasy_points = fantasy_points(data[(data['date'] < date) & ((data['batting_team'] == data[data['match_id'] == m]['bowling_team'].iloc[0]) | (data['bowling_team'] == data[data['match_id'] == m]['batting_team'].iloc[0]))], player)

        # Calculate running average:
        avg_score = total_score / total_balls_faced if total_balls_faced > 0 else 0
        avg_wickets_taken = total_wickets_taken / num_matches if num_matches > 0 else 0
        avg_fantasy_points = total_fantasy_points / num_matches if num_matches > 0 else 0
        avg_venue_fantasy_points = total_venue_fantasy_points / num_matches if num_matches > 0 else 0
        avg_against_opposition_fantasy_points = total_against_opposition_fantasy_points / num_matches if num_matches > 0 else 0
        avg_balls_faced = total_balls_faced /  num_matches if num_matches > 0 else 0
        avg_outs = total_outs / num_matches if num_matches > 0 else 0
        avg_fours = total_fours / num_matches if num_matches > 0 else 0
        avg_sixes = total_sixes / num_matches if num_matches > 0 else 0
        # TODO: byes legbyes shouldnt count in total_runs_conceded
        avg_runs_conceded = total_runs_conceded / total_balls_bowled if total_balls_bowled > 0 else 0
        avg_balls_bowled = total_balls_bowled / num_matches if num_matches > 0 else 0
        avg_catches = total_catches / num_matches if num_matches > 0 else 0
        avg_run_outs = total_run_outs / num_matches if num_matches > 0 else 0

        # Get a past 5 running average for the player
        past_matches = data[(data['date'] < date) & ((data['striker'] == player) | (data['bowler'] == player) | (data['fielder'] == player))].sort_values(by='date', ascending=False)['match_id'].unique()
        if len(past_matches) > 5:
            # Get the last 5 matches
            past_matches = past_matches[:5]
        # Get all matches if less than 5
        total_score_5 = data[(data['match_id'].isin(past_matches)) & (data['striker'] == player)]['runs_of_bat'].sum()
        total_balls_faced_5 = data[(data['match_id'].isin(past_matches)) & (data['striker'] == player)].shape[0]
        total_balls_bowled_5 = data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)].shape[0]
        total_outs_5 = data[(data['match_id'].isin(past_matches)) & (data['striker'] == player)]['player_dismissed'].notna().sum()
        total_runs_conceded_5 = data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)]['runs_of_bat'].sum() + data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)]['extras'].sum()
        total_wickets_taken_5 = data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)]['player_dismissed'].notna().sum()

        avg_score_5 = total_score_5 / total_balls_faced_5 if total_balls_faced_5 > 0 else 0
        avg_balls_faced_5 = total_balls_faced_5 /  len(past_matches) if len(past_matches) > 0 else 0
        avg_outs_5 = total_outs_5 / len(past_matches) if len(past_matches) > 0 else 0
        avg_catches_5 = data[(data['match_id'].isin(past_matches)) & (data['fielder'] == player)]['wicket_type'].eq('caught').sum() / len(past_matches) if len(past_matches) > 0 else 0
        avg_runs_conceded_5 = total_runs_conceded_5 / total_balls_bowled_5 if total_balls_bowled_5 > 0 else 0
        avg_balls_bowled_5 = total_balls_bowled_5 / len(past_matches) if len(past_matches) > 0 else 0
        avg_wickets_taken_5 = total_wickets_taken_5 / len(past_matches) if len(past_matches) > 0 else 0
        avg_fantasy_points_5 = fantasy_points(data[data['match_id'].isin(past_matches)], player)

        temp_df.append({
            'player': player,
            'match_id': m,
            'date': date,
            'fantasy_points': fp,
            'venue': data[data['match_id'] == m]['venue'].iloc[0],
            'batting_team': data[data['match_id'] == m]['batting_team'].iloc[0],
            'bowling_team': data[data['match_id'] == m]['bowling_team'].iloc[0],
            'avg_score': avg_score,
            'avg_balls_faced': avg_balls_faced,
            'avg_outs': avg_outs,
            'avg_runs_conceded': avg_runs_conceded,
            'avg_balls_bowled': avg_balls_bowled,
            'avg_wickets_taken': avg_wickets_taken,
            'avg_fantasy_points': avg_fantasy_points,
            'avg_venue_fantasy_points': avg_venue_fantasy_points,
            'avg_against_opposition_fantasy_points': avg_against_opposition_fantasy_points,
            'avg_score_5': avg_score_5,
            'avg_balls_faced_5': avg_balls_faced_5,
            'avg_outs_5': avg_outs_5,
            'avg_runs_conceded_5': avg_runs_conceded_5,
            'avg_balls_bowled_5': avg_balls_bowled_5,
            'avg_wickets_taken_5': avg_wickets_taken_5,
            'avg_fantasy_points_5': avg_fantasy_points_5,
            'matches_played': num_matches,
            'avg_fours': avg_fours,
            'avg_sixes': avg_sixes,
            'avg_catches': avg_catches,
            'avg_runouts': avg_run_outs,
        })

df = pd.DataFrame(temp_df)

In [6]:
df.to_csv('../data/all_years_player_match_agg.csv', index=False)

In [7]:
external_data = pd.read_csv('../data/aggregate_player_match_features_with_external_data.csv')
new_df = pd.merge(df, external_data[['player', 'match_id', 'temp_max', 'temp_min', 'precipitation', 'weather_code', 'is_batter', 'is_bowler', 'is_allrounder', 'is_wicketkeeper']], on=['player', 'match_id'], how='left')
new_df['role'] = new_df.apply(lambda row: "batter" if row['is_batter'] else "bowler" if row['is_bowler'] else "all-rounder" if row['is_allrounder'] else "wicketkeeper", axis=1)
new_df.drop(columns=['is_batter', 'is_bowler', 'is_allrounder', 'is_wicketkeeper'], inplace=True)
new_df.to_csv('../data/aggregate_player_match_features_with_external_data.csv', index=False)